# 03d — Predictions Analysis

**Purpose:** Analyze model predictions, error cases, and confusion patterns.

| Input | Output |
|---|---|
| `artifacts/features/*.npy` | Error visualizations → `results/figures/predictions/` |

**Runtime:** ~2 minutes (CPU only)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = Path('results')
FIG_DIR = RESULTS_DIR / 'figures' / 'predictions'
FEAT_DIR = Path('artifacts/features')
for d in [FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Normal', 'Pneumonia']

## 1. Load Data (when available)

In [ ]:
# This analysis requires predictions from the main notebook
# Expected files: y_true.npy, y_pred.npy, y_prob.npy

pred_files = list(RESULTS_DIR.glob('y_*.npy'))
print(f'Found {len(pred_files)} prediction files: {[f.name for f in pred_files]}')

if not pred_files:
    print('\nNo predictions found. Run main notebook first.')
    print('Expected: y_true.npy, y_pred.npy, y_prob.npy')

## 2. Error Analysis Template

In [ ]:
def analyze_predictions(y_true, y_pred, y_prob, name, save_dir):
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Left: Confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_xticklabels(CLASS_NAMES)
    axes[0].set_yticklabels(CLASS_NAMES)
    axes[0].set_title(f'{name} Confusion Matrix')
    
    # Right: ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, 'b-', lw=2, label='ROC')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
    axes[1].set_xlabel('FPR')
    axes[1].set_ylabel('TPR')
    axes[1].set_title(f'{name} ROC Curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_dir / f'{name}_evaluation.pdf', bbox_inches='tight')
    plt.show()
    
    # Print metrics
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
    acc = accuracy_score(y_true, y_pred)
    bal = balanced_accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    
    print(f'=== {name} ===')
    print(f'  Accuracy:    {acc:.4f}')
    print(f'  Bal. Acc:  {bal:.4f}')
    print(f'  AUC:       {auc:.4f}')
    
    return {'accuracy': acc, 'balanced_acc': bal, 'auc': auc}

# Analyze if predictions available
if len(pred_files) >= 3:
    y_true = np.load(RESULTS_DIR / 'y_true.npy')
    y_pred = np.load(RESULTS_DIR / 'y_pred.npy')
    y_prob = np.load(RESULTS_DIR / 'y_prob.npy')
    
    results = analyze_predictions(y_true, y_pred, y_prob, 'Model', FIG_DIR)
else:
    print('Skipping analysis (no prediction files)')

## 3. Error Case Analysis

In [ ]:
# Analyze error patterns when predictions available
def analyze_errors(y_true, y_pred, y_prob, save_dir):
    errors = y_true != y_pred
    n_errors = errors.sum()
    print(f'Total errors: {n_errors} / {len(y_true)} ({100*n_errors/len(y_true):.1f}%)')
    
    # Error types
    fp = (y_true == 0) & (y_pred == 1)  # False positive
    fn = (y_true == 1) & (y_pred == 0)  # False negative
    
    print(f'  False positives (Normal → Pneumonia): {fp.sum()}')
    print(f'  False negatives (Pneumonia → Normal): {fn.sum()}')
    
    # Probability distribution for errors
    fig, ax = plt.subplots(figsize=(7, 4))
    
    ax.hist(y_prob[~errors], bins=30, alpha=0.6, label='Correct', color='green')
    ax.hist(y_prob[errors], bins=30, alpha=0.6, label='Error', color='red')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Count')
    ax.set_title('Prediction Confidence: Correct vs Error')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'error_distribution.pdf', bbox_inches='tight')
    plt.show()
    print(f'saved → {FIG_DIR}/error_distribution.pdf')

if len(pred_files) >= 3:
    analyze_errors(y_true, y_pred, y_prob, FIG_DIR)

## 4. Summary

| Check | Status |
|:---|:---:|
| Predictions loaded | ✅ if available |
| Confusion matrix plotted | ✅ template |
| Error analysis | ✅ template |

**Note:** Requires main notebook to run and save predictions first.